# Module 03 — Lesson 07
# Scalars and Vectors

---

> "The temperature is 30°C." That's a single number — a **scalar**. "The wind is blowing at 20 km/h toward the northeast." That's a number AND a direction — a **vector**.

This distinction turns out to be the foundation of every machine learning model you'll ever build. A single row of your dataset — say, a student's `[hours_studied, exam_score, attendance_percent]` — is really just a vector: an ordered list of numbers describing a point in space. Once you see data this way, "how similar are two students?" becomes a geometry question: "how far apart are two points?" That's the idea this lesson builds, from scratch.

This lesson kicks off the **Linear Algebra** portion of the module — the language every dataset gets translated into before an ML model can touch it.

---
## 🎯 Learning Objectives

By the end of this lesson, you will be able to:

- Distinguish a scalar (a single number) from a vector (an ordered list of numbers)
- Represent a row of data as a vector, and understand what a "dimension" means
- Perform vector addition, subtraction, and scalar multiplication from scratch
- Compute a vector's magnitude (length) and understand its geometric meaning
- Compute the Euclidean distance between two vectors, and use it to compare data points
- Normalize a vector into a unit vector and explain why ML pipelines do this

---
## 1. What Is a Scalar? What Is a Vector?

- A **scalar** is a single number: temperature (30°C), your age (20), a price (₹500).
- A **vector** is an ordered list of numbers: wind (20 km/h, northeast), a GPS location (latitude, longitude), or a student's record `[hours_studied, score, attendance]`.

The number of components in a vector is its **dimension**. `[3, 4]` is a 2-dimensional vector. `[8, 85, 92]` is 3-dimensional. A dataset with 10 columns means every row is a 10-dimensional vector — hard to *picture*, but the math works identically to 2D and 3D.

In [ ]:
# A scalar: just one number
temperature = 30

# A vector: an ordered list of numbers — here, a student's record
# [hours_studied, exam_score, attendance_percent]
priya = [8, 85, 92]

print(f"Scalar (temperature)         : {temperature}")
print(f"Vector (Priya's record)      : {priya}")
print(f"Dimension of Priya's vector  : {len(priya)}")
print()
print("Component 0 (hours studied)  :", priya[0])
print("Component 1 (exam score)     :", priya[1])
print("Component 2 (attendance %)   :", priya[2])
print()
print("Every row of a dataset is a vector. Every column is a DIMENSION.")
print("This is exactly how scikit-learn sees your data internally.")

---
## 2. Visualising Vectors — 2D Arrows From the Origin

In 2D, a vector `[x, y]` can be drawn as an arrow starting at the origin `(0, 0)` and ending at the point `(x, y)`. This geometric picture — a vector as a *direction and length* — is what makes vector arithmetic intuitive before we generalise to higher dimensions (10 columns, 100 columns...) where drawing is no longer possible but the math stays the same.

In [ ]:
def ascii_vector_2d(vectors, labels, grid_size=21):
    '''Plot 2D vectors as lines from the origin on a small ASCII grid.'''
    half = grid_size // 2
    grid = [[" "] * grid_size for _ in range(grid_size)]
    grid[half][half] = "o"   # origin

    symbols = "ABCDEFG"
    for i, (vx, vy) in enumerate(vectors):
        steps = max(int(max(abs(vx), abs(vy)) * 4), 1)
        for s in range(steps + 1):
            t = s / steps
            col = half + round(vx * t)
            row = half - round(vy * t)   # y grows upward on screen
            if 0 <= row < grid_size and 0 <= col < grid_size and grid[row][col] == " ":
                grid[row][col] = symbols[i]
        end_col, end_row = half + round(vx), half - round(vy)
        if 0 <= end_row < grid_size and 0 <= end_col < grid_size:
            grid[end_row][end_col] = symbols[i]

    for row in grid:
        print("  " + "".join(row))
    for i, lab in enumerate(labels):
        print(f"  {symbols[i]} = {lab}")


v1 = [3, 4]
v2 = [-4, 2]

print("Two vectors, drawn as arrows from the origin 'o':")
ascii_vector_2d([v1, v2], [f"{v1}", f"{v2}"])

---
## 3. Vector Addition and Subtraction

Vectors are added (or subtracted) **component-wise** — each dimension combines independently with the matching dimension of the other vector. Geometrically, adding vectors is like placing them head-to-tail; the sum is the vector from the start of the first to the end of the second.

In [ ]:
def vector_add(a, b):
    '''Add two vectors component-wise. Both must be the same length.'''
    if len(a) != len(b):
        raise ValueError(f"Cannot add vectors of different dimensions: {len(a)} vs {len(b)}")
    return [ai + bi for ai, bi in zip(a, b)]

def vector_subtract(a, b):
    '''Subtract vector b from vector a, component-wise.'''
    if len(a) != len(b):
        raise ValueError(f"Cannot subtract vectors of different dimensions: {len(a)} vs {len(b)}")
    return [ai - bi for ai, bi in zip(a, b)]


v1 = [3, 4]
v2 = [-4, 2]

v_sum = vector_add(v1, v2)
v_diff = vector_subtract(v1, v2)

print(f"v1        = {v1}")
print(f"v2        = {v2}")
print(f"v1 + v2   = {v_sum}")
print(f"v1 - v2   = {v_diff}")
print()
ascii_vector_2d([v1, v2, v_sum], [f"v1={v1}", f"v2={v2}", f"v1+v2={v_sum}"])

---
## 4. Scalar Multiplication

Multiplying a vector by a scalar (a plain number) scales every component by that number. A positive scalar > 1 stretches the vector; a scalar between 0 and 1 shrinks it; a negative scalar reverses its direction.

In [ ]:
def scalar_multiply(k, v):
    '''Multiply every component of vector v by scalar k.'''
    return [k * vi for vi in v]


v = [2, 3]
v_stretched = scalar_multiply(2, v)     # stretch
v_shrunk    = scalar_multiply(0.5, v)   # shrink
v_reversed  = scalar_multiply(-1, v)    # reverse direction

print(f"v            = {v}")
print(f"2 * v        = {v_stretched}   (stretched, same direction)")
print(f"0.5 * v      = {v_shrunk}   (shrunk, same direction)")
print(f"-1 * v       = {v_reversed}   (reversed direction)")
print()
ascii_vector_2d([v, v_stretched, v_reversed], [f"v={v}", f"2v={v_stretched}", f"-v={v_reversed}"])

---
## 5. Vector Magnitude (Length)

The **magnitude** (or **norm**) of a vector is its length — the straight-line distance from the origin to its endpoint. It generalises the Pythagorean theorem to any number of dimensions:

$$\|v\| = \sqrt{v_1^2 + v_2^2 + \cdots + v_n^2}$$

Notice the "sum of squares, then square root" pattern — the exact same shape as the standard deviation formula from Lesson 02.

In [ ]:
import math

def magnitude(v):
    '''Euclidean length (norm) of a vector.'''
    return math.sqrt(sum(vi ** 2 for vi in v))


v1 = [3, 4]
v2 = [8, 85, 92]

print(f"||{v1}|| = {magnitude(v1):.2f}   (the classic 3-4-5 right triangle)")
print(f"||{v2}|| = {magnitude(v2):.2f}   (magnitude works in ANY dimension, not just 2D)")

---
## 6. Distance Between Two Vectors

The **Euclidean distance** between two vectors is the magnitude of their difference — how far apart their endpoints are:

$$d(a, b) = \|a - b\| = \sqrt{\sum_i (a_i - b_i)^2}$$

This is the foundation of "how similar are two data points?" — the core question behind K-Nearest Neighbors and clustering, which you'll meet in later modules.

In [ ]:
def distance(a, b):
    '''Euclidean distance between two vectors.'''
    return magnitude(vector_subtract(a, b))


priya = [8, 85, 92]   # [hours_studied, score, attendance%]
rohan = [3, 55, 70]
meera = [7, 80, 88]

print(f"distance(Priya, Rohan) = {distance(priya, rohan):.2f}")
print(f"distance(Priya, Meera) = {distance(priya, meera):.2f}")
print()
print("Priya and Meera are much closer in this 3D 'feature space' than")
print("Priya and Rohan — their study habits, scores, and attendance are")
print("more similar. This is exactly how a Nearest Neighbor model decides")
print("which historical data point is most like a new one.")

---
## 7. Unit Vectors — Normalization

A **unit vector** has magnitude exactly 1 — it keeps a vector's *direction* while discarding its *scale*. Divide every component by the magnitude:

$$\hat{v} = \frac{v}{\|v\|}$$

This is conceptually the same idea as z-score standardization from Lesson 02 — both remove "how big are the numbers" so that direction/shape, not scale, is what gets compared.

In [ ]:
def unit_vector(v):
    '''Normalize v to a unit vector (magnitude 1), preserving direction.'''
    m = magnitude(v)
    if m == 0:
        raise ValueError("Cannot normalize the zero vector (magnitude is 0)")
    return scalar_multiply(1 / m, v)


v = [3, 4]
u = unit_vector(v)

print(f"v          = {v}")
print(f"||v||      = {magnitude(v):.2f}")
print(f"unit(v)    = {[round(c, 4) for c in u]}")
print(f"||unit(v)|| = {magnitude(u):.4f}  <- should be 1.0")
print()
print("Same direction, length rescaled to exactly 1. In ML, feature vectors")
print("are often normalized like this (L2 normalization) so that no single")
print("feature dominates just because its raw numbers happen to be larger.")

---
## 8. Applied: Finding the Most Similar Data Point

In [ ]:
# A new student joins. Which existing student are they most similar to?
students = {
    "Priya": [8, 85, 92],
    "Rohan": [3, 55, 70],
    "Meera": [7, 80, 88],
    "Karan": [2, 50, 65],
}
new_student = [7.5, 82, 90]   # [hours_studied, score, attendance%]

print("Distance from the new student to each existing student:")
distances = {}
for name, vec in students.items():
    d = distance(new_student, vec)
    distances[name] = d
    print(f"  {name:<8}: {vec}  -> distance = {d:.2f}")

closest = min(distances, key=distances.get)
print()
print(f"Most similar student: {closest} (smallest distance)")
print()
print("This is the ENTIRE idea behind K-Nearest Neighbors: represent each")
print("data point as a vector, compute distances, and find the closest ones.")

---
## 9. Built-in Verification

In [ ]:
import math

v = [3, 4, 12]
a = [8, 85, 92]
b = [3, 55, 70]

# Python's math module has direct equivalents for magnitude and distance:
#   math.hypot(*v)   -> magnitude (works for any number of dimensions since 3.8)
#   math.dist(a, b)  -> Euclidean distance between two points

our_magnitude = magnitude(v)
stdlib_magnitude = math.hypot(*v)

our_distance = distance(a, b)
stdlib_distance = math.dist(a, b)

print(f"{'Function':<20} {'Ours':>12} {'stdlib':>12}  Match?")
print("-" * 52)
match_m = abs(our_magnitude - stdlib_magnitude) < 1e-9
match_d = abs(our_distance - stdlib_distance) < 1e-9
print(f"{'magnitude(v)':<20} {our_magnitude:>12.4f} {stdlib_magnitude:>12.4f}   {'match' if match_m else 'MISMATCH'}")
print(f"{'distance(a, b)':<20} {our_distance:>12.4f} {stdlib_distance:>12.4f}   {'match' if match_d else 'MISMATCH'}")

---
## ⚠️ Common Mistakes

In [ ]:
# -- Mistake 1: Adding vectors of different dimensions ------------------------

v1 = [1, 2, 3]
v2 = [4, 5]

print("Mistake 1 — Adding vectors of mismatched dimensions:")
try:
    vector_add(v1, v2)
except ValueError as e:
    print(f"  Caught error: {e}")
print()
print("  Vector addition/subtraction only makes sense when both vectors have")
print("  the SAME number of dimensions — component 0 must pair with component 0,")
print("  component 1 with component 1, and so on. Always check lengths first.")

In [ ]:
# -- Mistake 2: Thinking magnitude() returns a vector -------------------------

v = [3, 4]
result = magnitude(v)

print("Mistake 2 — Confusing a vector's magnitude with the vector itself:")
print(f"  v            = {v}          <- a vector (a list)")
print(f"  magnitude(v) = {result}          <- a SCALAR (a single number)")
print()
print("  magnitude() always collapses a vector down to ONE number describing")
print("  its length. It can never be used in place of the vector itself —")
print("  you lose all directional/component information.")

In [ ]:
# -- Mistake 3: Forgetting to guard against normalizing the zero vector -------

zero_vector = [0, 0, 0]

print("Mistake 3 — Normalizing the zero vector:")
try:
    unit_vector(zero_vector)
except ValueError as e:
    print(f"  Caught error: {e}")
print()
print("  The zero vector has magnitude 0 and no defined direction — dividing")
print("  by its magnitude is a division by zero. Always guard normalize()")
print("  functions against this, just like we did for z-scores in Lesson 02.")

In [ ]:
# -- Mistake 4: Mismatched component order between vectors --------------------

# WRONG: comparing vectors whose columns represent DIFFERENT things
student_a = [8, 85, 92]              # [hours_studied, score, attendance]
student_b_wrong_order = [92, 85, 8]  # accidentally [attendance, score, hours_studied]!

wrong_distance = distance(student_a, student_b_wrong_order)

# CORRECT: same student, columns in the SAME order
student_b_correct = [8, 85, 92]
correct_distance = distance(student_a, student_b_correct)

print("Mistake 4 — Vector components must represent the SAME feature in the SAME order:")
print(f"  student_a               = {student_a}")
print(f"  student_b (wrong order) = {student_b_wrong_order}")
print(f"  distance (bug!)         = {wrong_distance:.2f}   <- comparing hours to attendance!")
print()
print(f"  student_b (correct)     = {student_b_correct}")
print(f"  distance (correct)      = {correct_distance:.2f}")
print()
print("  If two datasets build their feature vectors with columns in different")
print("  orders, every distance/similarity computed between them is meaningless —")
print("  this is a silent bug that produces a NUMBER, not an error.")

---
## ✏️ Practice Exercises

### Exercise 1 — Starter: Vector Arithmetic

Given `a = [4, -2, 7]` and `b = [1, 5, -3]`, compute `a + b`, `a - b`, and `3 * a` using the functions from this lesson.

In [ ]:
a = [4, -2, 7]
b = [1, 5, -3]
# Your code here

### Exercise 2 — Magnitude and Unit Vector

For the vector `v = [6, -8]`, compute its magnitude and its unit vector. Verify the unit vector's magnitude is 1.0.

In [ ]:
v = [6, -8]
# Your code here

### Exercise 3 — Nearest Neighbor Product Recommender

You run a small store. Each product is represented as a vector `[price, rating, popularity_score]`. A customer just viewed a product — find the most similar OTHER product using `distance()`.

```python
products = {
    "Product A": [500, 4.2, 80],
    "Product B": [520, 4.5, 75],
    "Product C": [150, 3.8, 95],
    "Product D": [510, 4.1, 78],
}
viewed_product = [505, 4.3, 79]
```

In [ ]:
products = {
    "Product A": [500, 4.2, 80],
    "Product B": [520, 4.5, 75],
    "Product C": [150, 3.8, 95],
    "Product D": [510, 4.1, 78],
}
viewed_product = [505, 4.3, 79]
# Your code here

### Exercise 4 — Verify Against `math.dist` and `math.hypot`

Write a small test harness that generates 5 random 4-dimensional vector pairs (using `random.uniform(-10, 10)`), computes `distance()` and `magnitude()` for each, and checks they match `math.dist()` and `math.hypot()` within `1e-9`.

In [ ]:
import random
# Your code here

### Exercise 5 — (Challenge) Closest Pair

Write `closest_pair(vectors)` that takes a dict of `{name: vector}` and returns the pair of names with the SMALLEST distance between them (check every pair, not just distances to one target). Test it on:

```python
cities = {
    "CityA": [12.9, 77.6],
    "CityB": [13.1, 77.5],
    "CityC": [28.6, 77.2],
    "CityD": [19.1, 72.9],
}
```

In [ ]:
cities = {
    "CityA": [12.9, 77.6],
    "CityB": [13.1, 77.5],
    "CityC": [28.6, 77.2],
    "CityD": [19.1, 72.9],
}

def closest_pair(vectors):
    # Your code here
    pass

print(closest_pair(cities))

---
## 📌 Key Takeaways

- **A vector is just an ordered list of numbers, and a row of your dataset IS a vector.** Once you see data this way, every column becomes a dimension, and questions like "how similar are two records?" become geometry — a question about distance between points in space.

- **Magnitude and distance both use the "sum of squares, then square root" pattern you first saw with standard deviation in Lesson 02.** This isn't a coincidence — it's the same underlying idea (the Pythagorean theorem, generalized to any number of dimensions) showing up across statistics and linear algebra.

- **Normalizing a vector (dividing by its magnitude) isolates direction from scale — the geometric cousin of z-score standardization.** Both techniques exist for the same reason: so that raw scale differences between features don't distort comparisons or dominate a model.

---
## What's Next?

**Lesson 08 — Matrices and Matrix Multiplication**
You now know how to represent ONE data row as a vector. Next, you'll stack many rows together into a **matrix** — literally what a pandas DataFrame or a NumPy array becomes internally — and learn the matrix operations that power everything from image transformations to neural network layers.